In [0]:
# ============================================================
# CELL 1 — Install dependencies
# PyMuPDF (fitz) is pre-installed in DBR ML. pip will only
# install pymupdf4llm itself plus any genuinely missing deps.
# Watch the output: if pymupdf is listed as being UPGRADED
# (not just satisfied), cancel and confirm before proceeding.
# ============================================================
%pip install "pymupdf4llm>=0.0.17,<0.1.0" --quiet
dbutils.library.restartPython()

In [0]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import re
import hashlib
from datetime import datetime, timezone

import fitz                    # PyMuPDF — pre-installed in DBR ML
import pymupdf4llm             # Installed above

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType,
)
from delta.tables import DeltaTable

In [0]:
# ============================================================
# CELL 3 — Configuration
# Update all placeholder values before running.
# ============================================================

# --- Storage ---
ADLS_SOURCE_PATH   = "/Volumes/llmtest/exam_manual/pdfs"
# "abfss://<container>@<storage_account>.dfs.core.windows.net/<pdf_folder>/"
catalog = "llmtest"
schema = "exam_manual"
DELTA_OUTPUT_TABLE = f"{catalog}.{schema}.em_pdf_chunks"
# CHECKPOINT_PATH    = "abfss://<container>@<storage_account>.dfs.core.windows.net/_checkpoints/pdf_ingestion/"
# SCHEMA_LOCATION    = "abfss://<container>@<storage_account>.dfs.core.windows.net/_schema/pdf_ingestion/"

# --- Revision date stripping ---
# Matches an underscore followed by exactly 8 digits immediately before the file extension.
# Covers the pattern "HR_Manual_20260329.pdf" -> "HR_Manual"
# If your naming convention differs (e.g. "HR_Manual_2026-03-29.pdf") adjust accordingly.
REVISION_DATE_PATTERN: str = r"_\d{8}(?=\.\w+$)"

# --- Chunking ---
MAX_CHUNK_CHARS = 1500   # ~350 tokens; adjust to your embedding model's context window
OVERLAP_CHARS   = 150    # Overlap carried from end of one text chunk into the next

# --- Part mapping ---
# Maps lowercase filename substrings to part labels.
# First matching key wins. Add entries to match your naming convention.
# Example: "Safety_Management_Policy_v3.pdf" with key "safety" -> "Safety Management"
PART_MAPPING: dict[str, str] = {
     "1.": "General Information",
    "11.": "CAMELS-Based Examination",
    "21.": "CAMELS-Based Examination",
    "22.": "CAMELS-Based Examination",
    "31.": "CAMELS-Based Examination",
    "33.": "CAMELS-Based Examination",
    "41.": "CAMELS-Based Examination",
    "51.": "CAMELS-Based Examination",
    "61.": "CAMELS-Based Examination",
    "WP ": "Standard Examination Work"
}

# --- Category / topic extraction ---
# Regex patterns applied to the first-page text. Each must have exactly one capture group.
# Set either to None to use the fallback (first/second non-empty line of page 1).
# Run Cell 9 on sample files to inspect the raw text and determine the right patterns.
CATEGORY_PATTERN: str | None = None   # e.g. r"^Category[:\s]+(.+)$"
TOPIC_PATTERN:    str | None = None   # e.g. r"^Topic[:\s]+(.+)$"

# --- Multi-page table detection thresholds (in PDF points; 1pt ≈ 0.35mm) ---
# Increase TABLE_BOTTOM_THRESHOLD if multi-page tables are not being detected.
# Decrease if unrelated tables on adjacent pages are being incorrectly stitched.
TABLE_BOTTOM_THRESHOLD: float = 60.0
TABLE_TOP_THRESHOLD:    float = 80.0

In [0]:
# ============================================================
# CELL 4 — Output table schema
# ============================================================
CHUNK_SCHEMA = StructType([
    # Identity
    StructField("chunk_id",          StringType(),    False),  # SHA-256 of path + position
    StructField("content_hash",      StringType(),    False),  # SHA-256 of chunk_text; detects content changes without re-embedding
    # Document hierarchy
    StructField("part",              StringType(),    True),   # From PART_MAPPING
    StructField("category",          StringType(),    True),   # From top of PDF
    StructField("topic",             StringType(),    True),   # From top of PDF
    # File
    StructField("file_name",         StringType(),    False),
    StructField("file_path",         StringType(),    False),
    StructField("document_root_name", StringType(),    False),
    # Citation
    StructField("page_range_start",  IntegerType(),   True),   # 1-indexed
    StructField("page_range_end",    IntegerType(),   True),   # Equals start for single-page chunks
    StructField("page_citation",     StringType(),    True),   # "p. 5" or "pp. 3–5"
    # Structural context
    StructField("section",           StringType(),    True),   # Nearest H1/H2 above this chunk
    StructField("subsection",        StringType(),    True),   # Nearest H3/H4 above this chunk
    StructField("chunk_index",       IntegerType(),   False),  # Ordinal within file
    StructField("chunk_type",        StringType(),    True),   # "text" or "table"
    # Content
    StructField("chunk_text",        StringType(),    False),
    StructField("char_count",        IntegerType(),   True),
    # Lineage
    StructField("ingested_at",       TimestampType(), False),
    StructField("source_modified",   TimestampType(), True),
    StructField("version",    IntegerType(), False),  # Increments each time the source PDF changes
    StructField("is_current", StringType(),  False),  # "true" or "false" — used for RAG filtering
    StructField("parser_used",       StringType(),    True),
])

In [0]:
# ============================================================
# CELL 5 — Document-level metadata utilities
# ============================================================

def get_document_root_name(file_name: str, revision_date_pattern: str) -> str:
    """
    Strip the revision date suffix from a filename to produce a stable root name.

    Examples:
        "HR_Manual_20260329.pdf"  -> "HR_Manual"
        "Safety_Policy_20251201.pdf" -> "Safety_Policy"
        "Finance_Guide.pdf"       -> "Finance_Guide"  (no date suffix, unchanged)

    The .pdf extension is always removed from the returned value.
    If no date pattern matches, the bare filename without extension is returned,
    so documents without a date suffix are handled gracefully.
    """
    root = re.sub(revision_date_pattern, "", file_name)  # e.g. "HR_Manual.pdf"
    root = re.sub(r"\.\w+$", "", root)                   # e.g. "HR_Manual"
    return root
    

def get_part(file_name: str, part_mapping: dict[str, str]) -> str | None:
    """
    Return the part label for a filename by scanning PART_MAPPING for the first
    matching substring (case-insensitive) in the text before '.pdf'. Returns None if no key matches.
    """
    match = re.search(r"(.*?)(?=\.pdf$)", file_name, re.IGNORECASE)
    if not match:
        return "PDF Title Error"
    pre_pdf_text = match.group(1).lower()
    for key, part in part_mapping.items():
        if pre_pdf_text.startswith(key.lower()):
            return part
    return "Unknown Part"


def extract_category_topic(
    doc: fitz.Document,
    category_pattern: str | None,
    topic_pattern: str | None,
) -> tuple[str | None, str | None]:
    """
    Extract category and topic from the first page of a PDF.
    Applies regex patterns if configured; falls back to the first two
    non-empty lines of page 1 if a pattern is None or produces no match.
    """
    if len(doc) == 0:
        return None, None

    first_page_text = doc[0].get_text("text")
    lines = [ln.strip() for ln in first_page_text.split("\n") if ln.strip()]

    category: str | None = None
    topic: str | None = None

    if category_pattern:
        m = re.search(category_pattern, first_page_text, re.IGNORECASE | re.MULTILINE)
        if m:
            category = m.group(1).strip()

    if topic_pattern:
        m = re.search(topic_pattern, first_page_text, re.IGNORECASE | re.MULTILINE)
        if m:
            topic = m.group(1).strip()

    if category is None:
        category = lines[0] if lines else None
    if topic is None:
        topic = lines[1] if len(lines) > 1 else None

    return category, topic

In [0]:
# ============================================================
# CELL 6 — Table extraction with multi-page stitching
# ============================================================

def _rows_to_markdown(rows: list[list]) -> str:
    """
    Convert a list of table rows to a GitHub-flavoured markdown table.
    None cells become empty strings. Pipe characters in cell content are escaped.
    The first row is treated as the header.
    """
    if not rows:
        return ""

    max_cols = max(len(row) for row in rows)
    if max_cols == 0:
        return ""

    def clean_cell(cell) -> str:
        if cell is None:
            return ""
        return str(cell).replace("\n", " ").replace("|", "\\|").strip()

    def pad(row: list) -> list:
        return row + [None] * (max_cols - len(row))

    lines: list[str] = []
    lines.append("| " + " | ".join(clean_cell(c) for c in pad(rows[0])) + " |")
    lines.append("| " + " | ".join("---" for _ in range(max_cols)) + " |")
    for row in rows[1:]:
        lines.append("| " + " | ".join(clean_cell(c) for c in pad(row)) + " |")

    return "\n".join(lines)


def extract_tables_with_stitching(
    doc: fitz.Document,
    bottom_threshold: float = TABLE_BOTTOM_THRESHOLD,
    top_threshold: float = TABLE_TOP_THRESHOLD,
) -> list[dict]:
    """
    Extract all tables from the document and stitch multi-page tables together.

    Stitching rules:
      - A table on page N is a continuation candidate if its bottom edge falls within
        `bottom_threshold` points of the page bottom.
      - It is confirmed as a continuation if the first unprocessed table on page N+1
        starts within `top_threshold` points of the page top AND shares the same
        column count.
      - If the continuation table's first row matches the original header, it is
        treated as a repeated header and dropped.

    Returns a list of table dicts:
        {
            "start_page":   int,       # 1-indexed
            "end_page":     int,       # 1-indexed; equals start_page for single-page tables
            "rows":         list[list],
            "markdown":     str,
            "is_multipage": bool,
        }
    """
    # --- Pass 1: collect per-page table info ---
    per_page: dict[int, list[dict]] = {}

    for page_idx in range(len(doc)):
        page = doc[page_idx]
        page_height = page.rect.height
        page_num = page_idx + 1

        try:
            tab_finder = page.find_tables()
        except Exception as exc:
            print(f"[WARN] find_tables() failed on page {page_num}: {exc}")
            per_page[page_num] = []
            continue

        page_tables: list[dict] = []

        for tab in tab_finder.tables:
            rows = tab.extract()
            if not rows:
                continue
            # Skip degenerate 1×1 tables — usually false positives from simple bordered cells
            if len(rows) == 1 and (not rows[0] or len(rows[0]) <= 1):
                continue

            x0, y0, x1, y1 = tab.bbox
            col_count = max(len(row) for row in rows)

            page_tables.append({
                "page_num":       page_num,
                "rows":           rows,
                "col_count":      col_count,
                "touches_bottom": y1 >= page_height - bottom_threshold,
                "touches_top":    y0 <= top_threshold,
                "y0":             y0,
            })

        page_tables.sort(key=lambda t: t["y0"])
        per_page[page_num] = page_tables

    # --- Pass 2: stitch multi-page tables ---
    consumed: set[tuple[int, int]] = set()
    result: list[dict] = []

    for page_num in sorted(per_page.keys()):
        for tab_idx, tab in enumerate(per_page[page_num]):
            if (page_num, tab_idx) in consumed:
                continue

            consumed.add((page_num, tab_idx))
            combined_rows: list[list] = list(tab["rows"])
            start_page = page_num
            end_page = page_num
            current = tab
            next_page_num = page_num + 1

            while current["touches_bottom"] and next_page_num in per_page:
                matched_idx: int | None = None
                matched_tab: dict | None = None

                for nidx, ntab in enumerate(per_page[next_page_num]):
                    if (next_page_num, nidx) in consumed:
                        continue
                    if ntab["touches_top"] and ntab["col_count"] == current["col_count"]:
                        matched_idx = nidx
                        matched_tab = ntab
                        break

                if matched_idx is None or matched_tab is None:
                    break

                consumed.add((next_page_num, matched_idx))
                end_page = next_page_num
                cont_rows = matched_tab["rows"]
                original_header = combined_rows[0] if combined_rows else []

                # Drop the repeated header if the PDF renderer re-prints it on each page
                if cont_rows and cont_rows[0] == original_header:
                    combined_rows.extend(cont_rows[1:])
                else:
                    combined_rows.extend(cont_rows)

                current = matched_tab
                next_page_num += 1

            result.append({
                "start_page":   start_page,
                "end_page":     end_page,
                "rows":         combined_rows,
                "markdown":     _rows_to_markdown(combined_rows),
                "is_multipage": end_page > start_page,
            })

    return result

In [0]:
# ============================================================
# CELL 7 — Chunking utilities
# ============================================================

def _make_chunk_id(file_path: str, label: str, idx: int) -> str:
    """Deterministic SHA-256 chunk ID. Stable across re-runs for the same position."""
    return hashlib.sha256(f"{file_path}||{label}||{idx}".encode()).hexdigest()


def _make_content_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


def _make_page_citation(start: int, end: int) -> str:
    return f"p. {start}" if start == end else f"pp. {start}\u2013{end}"


def _strip_markdown_tables(text: str) -> str:
    """
    Remove markdown table rows from pymupdf4llm output.
    A table row is a line whose stripped form both starts and ends with '|'.
    Also removes the single blank line immediately following a table block,
    and collapses 3+ consecutive blank lines to 2.
    """
    lines = text.split("\n")
    output: list[str] = []
    skip_next_blank = False

    for line in lines:
        stripped = line.strip()
        is_table_row = (
            stripped.startswith("|")
            and stripped.endswith("|")
            and len(stripped) > 2
        )
        if is_table_row:
            skip_next_blank = True
            continue
        if skip_next_blank and stripped == "":
            skip_next_blank = False
            continue
        skip_next_blank = False
        output.append(line)

    result = "\n".join(output)
    result = re.sub(r"\n{3,}", "\n\n", result)
    return result.strip()


def _update_section_state(
    text: str,
    section: str | None,
    subsection: str | None,
) -> tuple[str | None, str | None]:
    """
    Scan heading lines in `text` and return updated (section, subsection).
    H1/H2 updates section and resets subsection to None.
    H3/H4 updates subsection only.
    Returns state after the LAST heading found in the block.
    """
    for line in text.split("\n"):
        line = line.strip()
        m12 = re.match(r"^#{1,2}\s+(.+)$", line)
        m34 = re.match(r"^#{3,4}\s+(.+)$", line)
        if m12:
            section = m12.group(1).strip()
            subsection = None
        elif m34:
            subsection = m34.group(1).strip()
    return section, subsection


def _build_text_chunk(
    text: str,
    page_num: int,
    section: str | None,
    subsection: str | None,
    file_path: str,
    file_name: str,
    document_root_name: str,
    part: str | None,
    category: str | None,
    topic: str | None,
    chunk_index: int,
    ingested_at: datetime,
    source_modified,
    version: int,
) -> dict:
    """Construct a single text chunk dict with all schema fields populated."""
    return {
        "chunk_id":           _make_chunk_id(file_path, f"text_p{page_num}", chunk_index),
        "content_hash":       _make_content_hash(text),
        "part":               part,
        "category":           category,
        "topic":              topic,
        "file_name":          file_name,
        "file_path":          file_path,
        "document_root_name": document_root_name,
        "page_range_start":   page_num,
        "page_range_end":     page_num,
        "page_citation":      _make_page_citation(page_num, page_num),
        "section":            section,
        "subsection":         subsection,
        "chunk_index":        chunk_index,
        "chunk_type":         "text",
        "chunk_text":         text,
        "char_count":         len(text),
        "ingested_at":        ingested_at,
        "source_modified":    source_modified,
        "parser_used":        "pymupdf4llm",
        "version":            version,
        "is_current":         "true",
    }


def chunk_text_page(
    page_text: str,
    page_num: int,
    file_path: str,
    file_name: str,
    document_root_name: str,
    part: str | None,
    category: str | None,
    topic: str | None,
    section: str | None,
    subsection: str | None,
    chunk_start_idx: int,
    ingested_at: datetime,
    source_modified,
    max_chars: int,
    overlap_chars: int,
    version: int,
) -> tuple[list[dict], int, str | None, str | None]:
    """
    Chunk one page's markdown text after table rows have been stripped.

    Split strategy:
      1. Split on double-newlines (paragraph boundary).
      2. Force a new chunk when the section or subsection changes.
      3. Force a new chunk when adding the next paragraph would exceed max_chars.
      4. Hard-split by character for any single paragraph exceeding max_chars.

    Overlap: the last paragraph of a chunk is carried into the next chunk as a
    seed only when the section does NOT change at the boundary.

    Returns: (chunks, next_chunk_idx, section_at_end_of_page, subsection_at_end_of_page)
    """
    clean = _strip_markdown_tables(page_text)
    if not clean:
        return [], chunk_start_idx, section, subsection

    paragraphs = [p.strip() for p in re.split(r"\n\n+", clean) if p.strip()]

    idx = chunk_start_idx
    running_sec: str | None = section
    running_subsec: str | None = subsection
    chunk_sec: str | None = section
    chunk_subsec: str | None = subsection
    buf: list[str] = []
    buf_len: int = 0
    chunks: list[dict] = []

    def _flush() -> None:
        nonlocal idx
        if not buf:
            return
        text = "\n\n".join(buf)
        chunks.append(_build_text_chunk(
            text=text,
            page_num=page_num,
            section=chunk_sec,
            subsection=chunk_subsec,
            file_path=file_path,
            file_name=file_name,
            document_root_name=document_root_name,
            part=part,
            category=category,
            topic=topic,
            chunk_index=idx,
            ingested_at=ingested_at,
            source_modified=source_modified,
            version=version,
        ))
        idx += 1

    for para in paragraphs:
        new_sec, new_subsec = _update_section_state(para, running_sec, running_subsec)
        section_changed = (new_sec != running_sec or new_subsec != running_subsec)
        running_sec, running_subsec = new_sec, new_subsec

        # Case 1: oversized single paragraph — flush buffer then hard-split
        if len(para) > max_chars:
            _flush()
            buf = []
            buf_len = 0
            chunk_sec = running_sec
            chunk_subsec = running_subsec
            for i in range(0, len(para), max_chars - overlap_chars):
                sl = para[i : i + max_chars].strip()
                if sl:
                    chunks.append(_build_text_chunk(
                        text=sl,
                        page_num=page_num,
                        section=running_sec,
                        subsection=running_subsec,
                        file_path=file_path,
                        file_name=file_name,
                        document_root_name=document_root_name,
                        part=part,
                        category=category,
                        topic=topic,
                        chunk_index=idx,
                        ingested_at=ingested_at,
                        source_modified=source_modified,
                        version=version,
                    ))
                    idx += 1
            continue

        sep = 2 if buf else 0
        would_exceed = (buf_len + sep + len(para)) > max_chars

        # Case 2: flush required — section change or size limit reached
        if (section_changed or would_exceed) and buf:
            _flush()
            # Carry last paragraph as overlap only when staying in the same section
            if not section_changed and buf and len(buf[-1]) <= overlap_chars:
                overlap_para = buf[-1]
                buf = [overlap_para]
                buf_len = len(overlap_para)
            else:
                buf = []
                buf_len = 0
            chunk_sec = running_sec
            chunk_subsec = running_subsec
            sep = 2 if buf else 0

        buf.append(para)
        buf_len += sep + len(para)

    _flush()
    return chunks, idx, running_sec, running_subsec


def chunk_table(
    table: dict,
    file_path: str,
    file_name: str,
    document_root_name: str,
    part: str | None,
    category: str | None,
    topic: str | None,
    section: str | None,
    subsection: str | None,
    chunk_start_idx: int,
    ingested_at: datetime,
    source_modified,
    max_chars: int,
    version: int,
) -> tuple[list[dict], int]:
    """
    Chunk a table produced by extract_tables_with_stitching.
    If the full table exceeds max_chars, it is split into row groups.
    The header row is repeated at the top of every chunk for embedding context.

    Returns: (chunks, next_chunk_idx)
    """
    rows = table["rows"]
    if not rows or len(rows) < 2:
        return [], chunk_start_idx

    start_page = table["start_page"]
    end_page   = table["end_page"]
    header     = rows[0]
    body_rows  = rows[1:]
    idx        = chunk_start_idx
    chunks: list[dict] = []
    buf_body: list[list] = []

    def _flush_table(body: list[list]) -> None:
        nonlocal idx
        text = _rows_to_markdown([header] + body)
        chunks.append({
            "chunk_id":           _make_chunk_id(
                                      file_path,
                                      f"table_p{start_page}_{end_page}",
                                      idx,
                                  ),
            "content_hash":       _make_content_hash(text),
            "part":               part,
            "category":           category,
            "topic":              topic,
            "file_name":          file_name,
            "file_path":          file_path,
            "document_root_name": document_root_name,
            "page_range_start":   start_page,
            "page_range_end":     end_page,
            "page_citation":      _make_page_citation(start_page, end_page),
            "section":            section,
            "subsection":         subsection,
            "chunk_index":        idx,
            "chunk_type":         "table",
            "chunk_text":         text,
            "char_count":         len(text),
            "ingested_at":        ingested_at,
            "source_modified":    source_modified,
            "parser_used":        "pymupdf4llm+fitz_tables",
            "version":            version,
            "is_current":         "true",
        })
        idx += 1

    for row in body_rows:
        candidate_text = _rows_to_markdown([header] + buf_body + [row])
        if len(candidate_text) > max_chars and buf_body:
            _flush_table(buf_body)
            buf_body = [row]
        else:
            buf_body.append(row)

    if buf_body:
        _flush_table(buf_body)

    return chunks, idx


def pdf_bytes_to_chunks(
    pdf_bytes: bytes,
    file_path: str,
    source_modified,
    version: int,
    part_mapping: dict[str, str] = PART_MAPPING,
    category_pattern: str | None = CATEGORY_PATTERN,
    topic_pattern: str | None = TOPIC_PATTERN,
    max_chars: int = MAX_CHUNK_CHARS,
    overlap_chars: int = OVERLAP_CHARS,
    revision_date_pattern: str = REVISION_DATE_PATTERN,
) -> list[dict]:
    """
    Full pipeline for one PDF:
      1. Extract document-level metadata (part, category, topic, document_root_name)
      2. Extract and stitch tables (fitz)
      3. Extract per-page text as markdown (pymupdf4llm), stripping table rows
      4. Chunk text pages with section/subsection tracking
      5. Chunk tables with page-range citations

    Returns a flat list of chunk dicts conforming to CHUNK_SCHEMA.
    """
    file_name          = file_path.split("/")[-1]
    document_root_name = get_document_root_name(file_name, revision_date_pattern)
    ingested_at        = datetime.now(timezone.utc)

    doc = fitz.open(stream=pdf_bytes, filetype="pdf")

    try:
        part               = get_part(file_name, part_mapping)
        category, topic    = extract_category_topic(doc, category_pattern, topic_pattern)
        tables             = extract_tables_with_stitching(doc)
        raw_page_chunks    = pymupdf4llm.to_markdown(doc, page_chunks=True)
    finally:
        doc.close()

    # Build 1-indexed page_num -> markdown text map
    page_text_map: dict[int, str] = {}
    for item in raw_page_chunks:
        page_num_0 = item.get("metadata", {}).get("page", 0)
        text = item.get("text", "")
        if text.strip():
            page_text_map[page_num_0 + 1] = text

    # --- Chunk text pages in reading order ---
    all_chunks: list[dict] = []
    chunk_idx: int = 0
    running_sec: str | None = None
    running_subsec: str | None = None

    # Record section state at the START of each page for table attribution
    page_sec_state: dict[int, tuple[str | None, str | None]] = {}

    for page_num in sorted(page_text_map.keys()):
        page_sec_state[page_num] = (running_sec, running_subsec)

        text_chunks, chunk_idx, running_sec, running_subsec = chunk_text_page(
            page_text=page_text_map[page_num],
            page_num=page_num,
            file_path=file_path,
            file_name=file_name,
            document_root_name=document_root_name,
            part=part,
            category=category,
            topic=topic,
            section=running_sec,
            subsection=running_subsec,
            chunk_start_idx=chunk_idx,
            ingested_at=ingested_at,
            source_modified=source_modified,
            max_chars=max_chars,
            overlap_chars=overlap_chars,
            version=version,
        )
        all_chunks.extend(text_chunks)

    # --- Chunk tables ---
    for table in tables:
        start_page = table["start_page"]
        t_sec, t_subsec = page_sec_state.get(
            start_page, (running_sec, running_subsec)
        )

        table_chunks, chunk_idx = chunk_table(
            table=table,
            file_path=file_path,
            file_name=file_name,
            document_root_name=document_root_name,
            part=part,
            category=category,
            topic=topic,
            section=t_sec,
            subsection=t_subsec,
            chunk_start_idx=chunk_idx,
            ingested_at=ingested_at,
            source_modified=source_modified,
            max_chars=max_chars,
            version=version,
        )
        all_chunks.extend(table_chunks)

    return all_chunks

In [0]:
# ============================================================
# CELL 8 — Main pipeline function
# ============================================================

def pdf_bytes_to_chunks(
    pdf_bytes: bytes,
    file_path: str,
    source_modified,
    version: int,
    part_mapping: dict[str, str] = PART_MAPPING,
    category_pattern: str | None = CATEGORY_PATTERN,
    topic_pattern: str | None = TOPIC_PATTERN,
    max_chars: int = MAX_CHUNK_CHARS,
    overlap_chars: int = OVERLAP_CHARS,
    revision_date_pattern = REVISION_DATE_PATTERN,
) -> list[dict]:
    """
    Full pipeline for one PDF:
      1. Extract document-level metadata (part, category, topic)
      2. Extract and stitch tables (fitz)
      3. Extract per-page text as markdown (pymupdf4llm), stripping table rows
      4. Chunk text pages with section/subsection tracking
      5. Chunk tables with page-range citations

    Returns a flat list of chunk dicts conforming to CHUNK_SCHEMA.
    """
    file_name = file_path.split("/")[-1]
    document_root_name = get_document_root_name(          
        file_name, revision_date_pattern
    )
    ingested_at = datetime.now(timezone.utc)
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")

    try:
        part = get_part(file_name, part_mapping)
        category, topic = extract_category_topic(doc, category_pattern, topic_pattern)
        tables = extract_tables_with_stitching(doc)
        raw_page_chunks = pymupdf4llm.to_markdown(doc, page_chunks=True)
    finally:
        doc.close()

    # Build 1-indexed page_num -> markdown text map
    page_text_map: dict[int, str] = {}
    for item in raw_page_chunks:
        page_num_0 = item.get("metadata", {}).get("page", 0)
        text = item.get("text", "")
        if text.strip():
            page_text_map[page_num_0 + 1] = text

    # --- Chunk text pages in reading order ---
    all_chunks: list[dict] = []
    chunk_idx: int = 0
    running_sec: str | None = None
    running_subsec: str | None = None

    # Record the section state at the START of each page for table attribution
    page_sec_state: dict[int, tuple[str | None, str | None]] = {}

    for page_num in sorted(page_text_map.keys()):
        page_sec_state[page_num] = (running_sec, running_subsec)

        text_chunks, chunk_idx, running_sec, running_subsec = chunk_text_page(
            page_text=page_text_map[page_num],
            page_num=page_num,
            file_path=file_path,
            file_name=file_name,
            document_root_name=document_root_name,
            part=part,
            category=category,
            topic=topic,
            section=running_sec,
            subsection=running_subsec,
            chunk_start_idx=chunk_idx,
            ingested_at=ingested_at,
            source_modified=source_modified,
            max_chars=max_chars,
            overlap_chars=overlap_chars,
            version=version,
        )
        all_chunks.extend(text_chunks)

    # --- Chunk tables ---
    for table in tables:
        start_page = table["start_page"]
        # Assign the section state that was in effect at the start of the table's first page
        t_sec, t_subsec = page_sec_state.get(start_page, (running_sec, running_subsec))

        table_chunks, chunk_idx = chunk_table(
            table=table,
            file_path=file_path,
            file_name=file_name,
            document_root_name=document_root_name,
            part=part,
            category=category,
            topic=topic,
            section=t_sec,
            subsection=t_subsec,
            chunk_start_idx=chunk_idx,
            ingested_at=ingested_at,
            source_modified=source_modified,
            max_chars=max_chars,
            version=version,
        )
        all_chunks.extend(table_chunks)

    return all_chunks

In [0]:
# ============================================================
# CELL 9 — [Optional helper] Inspect first-page text
# Run this on a sample file to determine the correct
# CATEGORY_PATTERN and TOPIC_PATTERN values for your documents.
# ============================================================

def inspect_first_page(pdf_bytes: bytes, n_lines: int = 25) -> None:
    """Print the first n_lines of a PDF's first page, one line per row."""
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text = doc[0].get_text("text")
    doc.close()
    print("=== First-page raw text (top lines) ===")
    for i, line in enumerate(text.split("\n")[:n_lines], start=1):
        print(f"{i:>3}: {line!r}")


# Usage: read a sample file from ADLS and call inspect_first_page
# sample_row = (
#     spark.read.format("binaryFile")
#     .load("abfss://<container>@<storage>.dfs.core.windows.net/<sample>.pdf")
#     .first()
# )
# inspect_first_page(bytes(sample_row["content"]))

In [0]:
# ============================================================
# CELL 10 — Create output Delta table (if not already present)
# ============================================================
(spark.createDataFrame([], CHUNK_SCHEMA).write
    .format("delta")
    .mode("ignore")
    .partitionBy("document_root_name")
    .saveAsTable(DELTA_OUTPUT_TABLE))

print(f"Table ready: {DELTA_OUTPUT_TABLE}")

In [0]:
# ============================================================
# CELL 11 — foreachBatch handler
# ============================================================

def process_pdf_batch(batch_df, batch_id: int) -> None:
    """
    Called for each Auto Loader micro-batch.
    Each row in batch_df is one PDF file (path, content bytes, modificationTime).
    For every file in the batch:
      - All existing chunks for that file_name are deleted (handles updates cleanly)
      - Fresh chunks from pdf_bytes_to_chunks are inserted
    """
    if batch_df.isEmpty():
        return

    rows = batch_df.select("path", "content", "modificationTime").collect()
    print(f"[Batch {batch_id}] {len(rows)} file(s) to process")

    all_chunks: list[dict] = []
    processed_file_names: list[str] = []

    delta_table = DeltaTable.forName(spark, DELTA_OUTPUT_TABLE)

    for row in rows:
        file_path     = row["path"]
        pdf_bytes     = bytes(row["content"])
        modified_time = row["modificationTime"]
        file_name     = file_path.split("/")[-1]
        document_root_name = get_document_root_name(  
        file_name, REVISION_DATE_PATTERN
    )


        print(f"  Parsing: {file_name}")
        try:
            # --- Determine next version number for this file ---
            existing = (
                spark.table(DELTA_OUTPUT_TABLE)
                .filter(F.col("document_root_name") == document_root_name)
                .agg(F.max("version").alias("max_version"))
                .first()
            )
            current_version = existing["max_version"] if existing["max_version"] is not None else 0
            next_version = current_version + 1

            # --- Mark all existing chunks for this file as stale ---
            delta_table.update(
                condition=F.col("document_root_name") == document_root_name,
                set={"is_current": F.lit("false")}
            )

            # --- Parse and produce new chunks at next_version ---
            chunks = pdf_bytes_to_chunks(
                pdf_bytes=pdf_bytes,
                file_path=file_path,
                source_modified=modified_time,
                version=next_version,
            )
            all_chunks.extend(chunks)
            processed_file_names.append(file_name)

            text_chunks  = sum(1 for c in chunks if c["chunk_type"] == "text")
            table_chunks = sum(1 for c in chunks if c["chunk_type"] == "table")
            print(f"    -> v{next_version}: {text_chunks} text chunks, {table_chunks} table chunks")

        except Exception as exc:
            # On failure, do NOT leave the file in a stale state.
            # Roll the is_current flag back to "true" for the previous version.
            print(f"    [ERROR] {file_name}: {exc}")
            print(f"    [ROLLBACK] Restoring is_current=true for {file_name} v{current_version}")
            delta_table.update(
                condition=(F.col("document_root_name") == document_root_name) &  
                      (F.col("version") == current_version),
                set={"is_current": F.lit("true")}
            )

    if not all_chunks:
        print(f"[Batch {batch_id}] No chunks produced — skipping write")
        return

    chunks_df = spark.createDataFrame(all_chunks, schema=CHUNK_SCHEMA)
    chunks_df.write.format("delta").mode("append").saveAsTable(DELTA_OUTPUT_TABLE)

    print(f"[Batch {batch_id}] Wrote {len(all_chunks)} chunks to {DELTA_OUTPUT_TABLE}")

In [0]:
# ============================================================
# CELL 12b — POC: Read a single test PDF (no Auto Loader)
# Replace the path with the full ADLS path to your test file.
# This uses a standard batch read — no checkpoint or schema
# location needed.
# ============================================================

TEST_PDF_PATH = f"{ADLS_SOURCE_PATH}/1.3_20230622.pdf"

test_df = (
    spark.read
    .format("binaryFile")
    .load(TEST_PDF_PATH)
    .filter(F.col("length") > 0)
)

# Confirm the file was found before proceeding
print(f"Files found: {test_df.count()}")
display(test_df.select("path", "length", "modificationTime"))

In [0]:
# # ============================================================
# # Quick output inspection — verify extraction quality
# # ============================================================

# row = test_df.first()
# pdf_bytes = bytes(row["content"])
# doc = fitz.open(stream=pdf_bytes, filetype="pdf")

# # Check a specific page — adjust page_idx to a page you can
# # visually compare against the source PDF
# page_idx = 18
# page = doc[page_idx]

# print(f"=== Raw text extraction — page {page_idx + 1} ===")
# print(page.get_text("text"))
# print()

# doc.close()

In [0]:
# ============================================================
# CELL 13b — POC: Run the parsing pipeline on the test PDF
# Calls pdf_bytes_to_chunks directly, bypassing foreachBatch
# and all streaming infrastructure.
# ============================================================

row = test_df.first()

file_path     = row["path"]
pdf_bytes     = bytes(row["content"])
modified_time = row["modificationTime"]
file_name     = file_path.split("/")[-1]
document_root_name = get_document_root_name(file_name, REVISION_DATE_PATTERN)

print(f"Testing with: {file_name}")
print(f"Document root name: {document_root_name}")
print(f"File size: {row['length']:,} bytes")
print()

# Version 1 for a fresh POC run
chunks = pdf_bytes_to_chunks(
    pdf_bytes=pdf_bytes,
    file_path=file_path,
    source_modified=modified_time,
    version=1,
)

# --- Summary ---
text_chunks  = [c for c in chunks if c["chunk_type"] == "text"]
table_chunks = [c for c in chunks if c["chunk_type"] == "table"]

print(f"Total chunks:  {len(chunks)}")
print(f"Text chunks:   {len(text_chunks)}")
print(f"Table chunks:  {len(table_chunks)}")
print(f"Category:      {chunks[0]['category'] if chunks else 'n/a'}")
print(f"Topic:         {chunks[0]['topic'] if chunks else 'n/a'}")
print(f"Part:          {chunks[0]['part'] if chunks else 'n/a'}")
print()

# --- Write to Delta (same as process_pdf_batch would do) ---
# Marks any existing chunks for this document root as stale first,
# then inserts the new chunks. Safe to re-run — subsequent runs
# will increment the version and mark the previous run stale.
delta_table = DeltaTable.forName(spark, DELTA_OUTPUT_TABLE)

existing = (
    spark.table(DELTA_OUTPUT_TABLE)
    .filter(F.col("document_root_name") == document_root_name)
    .agg(F.max("version").alias("max_version"))
    .first()
)
current_version = existing["max_version"] if existing["max_version"] is not None else 0

if current_version > 0:
    print(f"Existing data found at v{current_version} — marking stale")
    delta_table.update(
        condition=F.col("document_root_name") == document_root_name,
        set={"is_current": F.lit("false")}
    )

chunks_df = spark.createDataFrame(chunks, schema=CHUNK_SCHEMA)
chunks_df.write.format("delta").mode("append").saveAsTable(DELTA_OUTPUT_TABLE)
print(f"Written to {DELTA_OUTPUT_TABLE} at v1")

In [0]:
spark.read.table("llmtest.exam_manual.em_pdf_chunks").display()

In [0]:
# # ============================================================
# # CELL 12 — Auto Loader stream definition
# # ============================================================
# raw_stream = (
#     spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format",              "binaryFile")
#     .option("cloudFiles.schemaLocation",      SCHEMA_LOCATION)
#     .option("pathGlobFilter",                 "*.pdf")
#     .option("cloudFiles.includeExistingFiles","true")  # Safe to leave permanently
#     .load(ADLS_SOURCE_PATH)
#     .filter(F.col("length") > 0)
# )

In [0]:
# # ============================================================
# # CELL 13 — Triggered execution
# # availableNow=True processes all pending files then terminates.
# # Schedule this cell (or the whole notebook) via Databricks
# # Workflows on whatever cadence matches your PDF update frequency.
# # ============================================================
# query = (
#     raw_stream.writeStream
#     .foreachBatch(process_pdf_batch)
#     .option("checkpointLocation", CHECKPOINT_PATH)
#     .trigger(availableNow=True)
#     .start()
# )

# query.awaitTermination()
# print("Ingestion run complete.")

In [0]:
# ============================================================
# CELL 14 — Validation queries
# ============================================================

# File-level summary with chunk type breakdown
# Version history now groups by document_root_name to show the full revision lineage — useful for audit review
display(
    spark.table(DELTA_OUTPUT_TABLE)
    .groupBy("document_root_name", "file_name", "version", "is_current")
    .agg(
        F.count("chunk_id").alias("chunk_count"),
        F.max("ingested_at").alias("ingested_at"),
    )
    .orderBy("document_root_name", "version")
)

In [0]:
# Inspect multi-page table chunks — verify page ranges and header repetition
display(
    spark.table(DELTA_OUTPUT_TABLE)
    .filter(
        (F.col("chunk_type") == "table") &
        (F.col("page_range_start") != F.col("page_range_end"))
    )
    .select(
        "file_name", "page_citation", "section", "subsection",
        "char_count", "chunk_text"
    )
    .orderBy("file_name", "page_range_start")
)